In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
print(os.getenv("GROQ_API_KEY"))

In [7]:
from ingest import load_currency_data
documents = load_currency_data()

In [8]:
import json

# Point directly to the single JSON file
file_path = './documents/currencies.json'

with open(file_path, 'r', encoding='utf-8') as f:
    documents = json.load(f)

documents

{'currencies': [{'code': 'USD',
   'name': 'US Dollar',
   'symbol': '$',
   'decimal_digits': 2,
   'countries': ['United States', 'Ecuador', 'El Salvador']},
  {'code': 'EUR',
   'name': 'Euro',
   'symbol': '€',
   'decimal_digits': 2,
   'countries': ['Germany', 'France', 'Italy', 'Spain', 'Netherlands']},
  {'code': 'GBP',
   'name': 'British Pound',
   'symbol': '£',
   'decimal_digits': 2,
   'countries': ['United Kingdom']},
  {'code': 'JPY',
   'name': 'Japanese Yen',
   'symbol': '¥',
   'decimal_digits': 0,
   'countries': ['Japan']},
  {'code': 'CHF',
   'name': 'Swiss Franc',
   'symbol': 'CHF',
   'decimal_digits': 2,
   'countries': ['Switzerland', 'Liechtenstein']},
  {'code': 'AUD',
   'name': 'Australian Dollar',
   'symbol': 'A$',
   'decimal_digits': 2,
   'countries': ['Australia']},
  {'code': 'CAD',
   'name': 'Canadian Dollar',
   'symbol': 'C$',
   'decimal_digits': 2,
   'countries': ['Canada']},
  {'code': 'CNY',
   'name': 'Chinese Yuan',
   'symbol': '¥',
 

In [9]:
from ingest import load_currency_data
documents = load_currency_data(file_path="./documents/currencies.json") # adjust path as needed

In [10]:
documents[10]

{'code': 'BRL',
 'name': 'Brazilian Real',
 'symbol': 'R$',
 'decimal_digits': 2,
 'countries': ['Brazil'],
 'doc_id': 'BRL',
 'countries_text': 'Brazil'}

In [11]:
doc = documents[25]

print("Document ID:", doc["doc_id"])
print("Currency code:", doc["code"])
print("Currency name:", doc["name"])
print("Currency symbol:", doc["symbol"])
print("Countries:", doc["countries_text"])

Document ID: PHP
Currency code: PHP
Currency name: Philippine Peso
Currency symbol: ₱
Countries: Philippines


In [12]:
doc

{'code': 'PHP',
 'name': 'Philippine Peso',
 'symbol': '₱',
 'decimal_digits': 2,
 'countries': ['Philippines'],
 'doc_id': 'PHP',
 'countries_text': 'Philippines'}

In [13]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [14]:
data_gen_instructions = """
Generate 5 natural, student-style questions based on a currency record.
Questions must be clear, concise, and answerable using the record's
fields, such as currency code, name, symbol, and countries.
Return only the questions.
""".strip()

In [15]:
data_gen_instructions

"Generate 5 natural, student-style questions based on a currency record.\nQuestions must be clear, concise, and answerable using the record's\nfields, such as currency code, name, symbol, and countries.\nReturn only the questions."

In [16]:
import json
user_prompt = json.dumps(doc)
user_prompt

'{"code": "PHP", "name": "Philippine Peso", "symbol": "\\u20b1", "decimal_digits": 2, "countries": ["Philippines"], "doc_id": "PHP", "countries_text": "Philippines"}'

In [17]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [18]:
doc

{'code': 'PHP',
 'name': 'Philippine Peso',
 'symbol': '₱',
 'decimal_digits': 2,
 'countries': ['Philippines'],
 'doc_id': 'PHP',
 'countries_text': 'Philippines'}

In [19]:
from rag_helper import RAGBase

In [20]:
from evaluation_utils import calc_price

c:\Users\Marun Yory\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [21]:
from openai import OpenAI
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [22]:
response = openai_client.responses.create(
    model="openai/gpt-oss-120b",
    input=messages,
)

In [23]:
response

Response(id='resp_01m62tk71peynajg10pzbebvq0', created_at=1793243847.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='openai/gpt-oss-120b', object='response', output=[ResponseReasoningItem(id='resp_01m62tk71peynveqr4111z5azs', summary=[], type='reasoning', content=[Content(text='The user gave a JSON record for currency PHP. Need to generate 5 natural student-style questions based on that record, using fields: code, name, symbol, decimal_digits, countries, doc_id, countries_text. Should be clear, concise, answerable using record\'s fields. Return only the questions, each on separate line presumably.\n\nExamples: "What is the currency code for the Philippine Peso?" "Which country uses the PHP currency?" "What symbol represents the Philippine Peso?" "How many decimal digits does the PHP have?" "What is the doc_id for the Philippine Peso?" Something like that.\n\nReturn only the questions, no extra text.', type='reasoning_text')], encrypted_content=None, statu

In [24]:
from evaluation_utils import llm_structured

In [25]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions,
    model="openai/gpt-oss-120b"
)

In [26]:
for i in (result.questions):
    print(i)

What is the three-letter code for the Philippine Peso?
Which country uses the Philippine Peso?
What symbol is used to denote the Philippine Peso?
How many decimal digits are typically used for the Philippine Peso?
What is the official name of the currency with the code PHP?


In [27]:
usage

ResponseUsage(input_tokens=276, input_tokens_details=InputTokensDetails(cache_write_tokens=None, cached_tokens=0), output_tokens=387, output_tokens_details=OutputTokensDetails(reasoning_tokens=313), total_tokens=663)

In [28]:
from evaluation_utils import calc_price

In [29]:
calc_price(usage)

{'input_cost': 2.76e-05, 'output_cost': 0.0001548, 'total_cost': 0.0001824}

In [30]:
doc

{'code': 'PHP',
 'name': 'Philippine Peso',
 'symbol': '₱',
 'decimal_digits': 2,
 'countries': ['Philippines'],
 'doc_id': 'PHP',
 'countries_text': 'Philippines'}

In [31]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [32]:
print(result)
print(result.model_dump())
print(result.model_fields)

questions=['What is the three-letter code for the Philippine Peso?', 'Which country uses the Philippine Peso?', 'What symbol is used to denote the Philippine Peso?', 'How many decimal digits are typically used for the Philippine Peso?', 'What is the official name of the currency with the code PHP?']
{'questions': ['What is the three-letter code for the Philippine Peso?', 'Which country uses the Philippine Peso?', 'What symbol is used to denote the Philippine Peso?', 'How many decimal digits are typically used for the Philippine Peso?', 'What is the official name of the currency with the code PHP?']}
{'questions': FieldInfo(annotation=list[str], required=True)}


C:\Users\Marun Yory\AppData\Local\Temp\ipykernel_10116\2250340996.py:3: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  print(result.model_fields)


In [33]:
records = []

for q in result.questions:
    q = q.strip().lstrip("0123456789.-) ")

    if q:
        records.append(q)

In [34]:
import pandas as pd
pd.DataFrame(records)

,0
0,What is the three-letter code for the Philippi...
1,Which country uses the Philippine Peso?
2,What symbol is used to denote the Philippine P...
3,How many decimal digits are typically used for...
4,What is the official name of the currency with...


In [35]:
from evaluation_utils import llm_structured_retry

In [36]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions,
        model="openai/gpt-oss-120b"
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["doc_id"]
        })

    return results, usage

In [37]:
doc

{'code': 'PHP',
 'name': 'Philippine Peso',
 'symbol': '₱',
 'decimal_digits': 2,
 'countries': ['Philippines'],
 'doc_id': 'PHP',
 'countries_text': 'Philippines'}

In [38]:
generate_ground_truth(doc)

([{'question': 'What is the currency code for the Philippine Peso?',
   'document': 'PHP'},
  {'question': 'Which country uses the currency with the symbol ₱?',
   'document': 'PHP'},
  {'question': 'How many decimal digits does the Philippine Peso have?',
   'document': 'PHP'},
  {'question': 'What is the full name of the currency coded as PHP?',
   'document': 'PHP'},
  {'question': 'In which country would you find the currency symbol ₱ in daily transactions?',
   'document': 'PHP'}],
 ResponseUsage(input_tokens=276, input_tokens_details=InputTokensDetails(cache_write_tokens=None, cached_tokens=0), output_tokens=380, output_tokens_details=OutputTokensDetails(reasoning_tokens=298), total_tokens=656))

In [ ]:
documents[:5]

In [40]:
data_gen_instructions

"Generate 5 natural, student-style questions based on a currency record.\nQuestions must be clear, concise, and answerable using the record's\nfields, such as currency code, name, symbol, and countries.\nReturn only the questions."

In [41]:
len(documents[:5])

5

In [42]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

100%|██████████| 5/5 [00:06<00:00,  1.36s/it]


In [ ]:
ground_truth

In [44]:
len(ground_truth)
# pd.DataFrame(ground_truth)

25

In [45]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [46]:
len(documents)

30

In [47]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=1) as pool:
    results = map_progress(
        pool,
        documents,
        generate_ground_truth
    )

100%|██████████| 30/30 [01:36<00:00,  3.21s/it]


In [48]:
openai_client.responses.create

<bound method Responses.create of <openai.resources.responses.responses.Responses object at 0x0000012B43999BE0>>

In [49]:
with ThreadPoolExecutor(max_workers=1) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

100%|██████████| 30/30 [02:25<00:00,  4.84s/it]


In [50]:
len(results)

30

In [51]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

150

In [52]:
usage

ResponseUsage(input_tokens=274, input_tokens_details=InputTokensDetails(cache_write_tokens=None, cached_tokens=0), output_tokens=623, output_tokens_details=OutputTokensDetails(reasoning_tokens=539), total_tokens=897)

In [53]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.005485900000000001

In [54]:
from evaluation_utils import calc_total_price
calc_total_price(usages)

0.0054859

In [55]:
import pandas as pd
df_ground_truth = pd.DataFrame(ground_truth)

In [56]:
df_ground_truth

,question,document
0,What is the three-letter code for the US Dollar?,USD
1,Which symbol is used to represent the US Dollar?,USD
2,How many decimal digits are typically shown fo...,USD
3,Name three countries where the US Dollar is us...,USD
4,What is the doc_id associated with the US Doll...,USD
...,...,...
145,What is the three-letter code for the Saudi Ri...,SAR
146,Which country uses the currency with the symbo...,SAR
147,What is the name of the currency that has the ...,SAR
148,How many decimal digits does the Saudi Riyal h...,SAR


In [57]:
from pathlib import Path

output_path = Path("../data/ground_truth-new.csv")

# Create the directory if it doesn't exist
output_path.parent.mkdir(parents=True, exist_ok=True)

# Save the DataFrame
df_ground_truth.to_csv(output_path, index=False)

print(f"File saved to: {output_path.resolve()}")

File saved to: C:\Users\Marun Yory\Desktop\data\ground_truth-new.csv


In [58]:
df_ground_truth.to_csv("../data/ground_truth-new.csv", index=False)

In [59]:
len(ground_truth)

150

In [60]:
import pandas as pd

df_ground_truth = pd.read_csv("../data/ground_truth-new.csv") #-data.csv")

In [84]:
df_ground_truth.head()

,question,document
0,What is the currency code for the US Dollar?,USD
1,Which symbol represents the US Dollar?,USD
2,How many decimal digits does the US Dollar use?,USD
3,Which countries use the US Dollar as their off...,USD
4,What is the full name of the currency with the...,USD


In [62]:
ground_truth = df_ground_truth.to_dict(orient="records")

In [63]:
from ingest import load_currency_data, build_index

# Load currency records
documents = load_currency_data(
    file_path="./documents/currencies.json"
)

# Build the currency search index
index = build_index(documents)

print(f"Loaded {len(documents)} currency records.")
print("Currency index created successfully.")

Loaded 30 currency records.
Currency index created successfully.


In [64]:
from ingest import load_currency_data, build_index

# Load currency records
documents = load_currency_data(
    file_path="./documents/currencies.json"
)

# Keep only records containing all required fields
documents_currency = []

for doc in documents:
    if all(
        key in doc
        for key in [
            "code",
            "name",
            "symbol",
            "decimal_digits",
            "countries",
        ]
    ):
        documents_currency.append(doc)

# Build the currency search index
index = build_index(documents_currency)

print(f"Loaded {len(documents)} currency records.")
print(f"Indexed {len(documents_currency)} valid currency records.")

Loaded 30 currency records.
Indexed 30 valid currency records.


In [65]:
index = build_index(documents)

# Boost currency-related fields
boost = {
    "code": 3.0,
    "name": 2.0,
    "symbol": 2.0,
    "countries_text": 2.0,
}

# Search for currencies
results = index.search(
    "What currency is used in Germany?",
    num_results=5,
    boost_dict=boost
)

# Display search results
for result in results:
    print(result)

{'code': 'EUR', 'name': 'Euro', 'symbol': '€', 'decimal_digits': 2, 'countries': ['Germany', 'France', 'Italy', 'Spain', 'Netherlands'], 'doc_id': 'EUR', 'countries_text': 'Germany, France, Italy, Spain, Netherlands'}


In [66]:
index = build_index(documents)

# Boost the currency name field
boost = {"name": 3.0}

# Search for currencies
results = index.search(
    "What currency is used in Germany?",
    num_results=5,
    boost_dict=boost
)

# Display results
for result in results:
    print(result)

{'code': 'EUR', 'name': 'Euro', 'symbol': '€', 'decimal_digits': 2, 'countries': ['Germany', 'France', 'Italy', 'Spain', 'Netherlands'], 'doc_id': 'EUR', 'countries_text': 'Germany, France, Italy, Spain, Netherlands'}


In [67]:
def text_search(query):
    boost_dict = {
        "code": 3.0,
        "name": 2.0,
        "symbol": 2.0,
        "countries_text": 2.0,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [68]:
q = ground_truth[1]
q

{'question': 'Which symbol is used to represent the US Dollar?',
 'document': 'USD'}

In [69]:
doc_id = q['document']
doc_id

'USD'

In [ ]:
results = text_search(q['question'])
results

In [71]:
for d in results:
    print(f'{d["doc_id"]} == {doc_id}: {d["doc_id"] == doc_id}')

USD == USD: True
AUD == USD: False
CAD == USD: False
SGD == USD: False
HKD == USD: False


In [72]:
relevance = []

for d in results:
    relevance.append(int(d["doc_id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [73]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["doc_id"] == doc_id))

    return relevance

In [74]:
q = ground_truth[10]
print(q["question"])
compute_relevance_text(q)

What is the currency code for the British Pound?


[1]

In [75]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [76]:
relevance = compute_relevance_total_text(ground_truth)

100%|██████████| 150/150 [00:00<00:00, 239.34it/s]


In [77]:
relevance[15:30]

[[1],
 [1],
 [1],
 [1],
 [1],
 [1],
 [1],
 [1],
 [1],
 [1],
 [1, 0, 0, 0, 0],
 [],
 [1],
 [1, 0, 0, 0, 0],
 [1]]

In [78]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["doc_id"] == doc_id))

    return relevance

In [79]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [80]:
relevance_total = compute_relevance_total(ground_truth, text_search)

100%|██████████| 150/150 [00:00<00:00, 227.39it/s]


In [81]:
sample = relevance_total[:15]
sample

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1],
 [1],
 [1],
 [1],
 [1],
 [1],
 [1],
 [1],
 [1],
 [1]]

In [82]:
cnt = 0

for line in sample:
    if 1 in line:
        cnt = cnt + 1

cnt / len(sample)

1.0

In [83]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [84]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [85]:
hit_rate(sample)

1.0

In [86]:
total_score = 0.0

for line in sample:
    for rank in range(len(line)):
        if line[rank] == 1:
            score = 1 / (rank + 1)
            total_score = total_score + score
            break

total_score / len(sample)

1.0

In [87]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)

In [88]:
mrr(sample)

1.0

In [89]:
len(relevance), len(ground_truth)

(150, 150)

In [90]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [91]:
evaluate(ground_truth, text_search) # vector_search, hybrid_search

100%|██████████| 150/150 [00:00<00:00, 223.96it/s]


{'hit_rate': 0.94, 'mrr': 0.9322222222222221}

In [92]:
# minsearch library

def text_search_v2(query):
    boost_dict = {
        "code": 1.0,
        "name": 3.0,
        "symbol": 2.0,
        "countries_text": 3.0,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

evaluate(ground_truth, text_search_v2)

100%|██████████| 150/150 [00:00<00:00, 204.28it/s]


{'hit_rate': 0.94, 'mrr': 0.9322222222222221}

In [93]:
def search_boost(query, name_boost):
    boost_dict = {
        "code": 1.0,
        "name": name_boost,
        "symbol": 2.0,
        "countries_text": 2.0,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [94]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

100%|██████████| 150/150 [00:00<00:00, 154.79it/s]


boost=0.5: {'hit_rate': 0.94, 'mrr': 0.9322222222222221}


100%|██████████| 150/150 [00:00<00:00, 199.40it/s]


boost=1.0: {'hit_rate': 0.94, 'mrr': 0.9322222222222221}


100%|██████████| 150/150 [00:00<00:00, 178.59it/s]


boost=3.0: {'hit_rate': 0.94, 'mrr': 0.9322222222222221}


100%|██████████| 150/150 [00:00<00:00, 199.87it/s]


boost=5.0: {'hit_rate': 0.94, 'mrr': 0.9322222222222221}


100%|██████████| 150/150 [00:00<00:00, 192.05it/s]

boost=10.0: {'hit_rate': 0.94, 'mrr': 0.9322222222222221}


In [95]:
def search_boosts(query, code_boost, name_boost, symbol_boost, countries_boost):
    boost_dict = {
        "code": code_boost,
        "name": name_boost,
        "symbol": symbol_boost,
        "countries_text": countries_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [ ]:
results = []

for code_boost in [1.0, 2.0, 5.0]:
    for name_boost in [1.0, 2.0, 4.0, 10.0]:
        for symbol_boost in [0.5, 1.0, 2.0]:
            for countries_boost in [1.0, 2.0, 5.0]:
                print(
                    f"Evaluating code_boost={code_boost}, "
                    f"name_boost={name_boost}, "
                    f"symbol_boost={symbol_boost}, "
                    f"countries_boost={countries_boost}..."
                )

                result = evaluate(
                    ground_truth,
                    lambda query,
                    code_boost=code_boost,
                    name_boost=name_boost,
                    symbol_boost=symbol_boost,
                    countries_boost=countries_boost: search_boosts(
                        query,
                        code_boost,
                        name_boost,
                        symbol_boost,
                        countries_boost,
                    )
                )

                results.append({
                    "code": code_boost,
                    "name": name_boost,
                    "symbol": symbol_boost,
                    "countries_text": countries_boost,
                    "hit_rate": result["hit_rate"],
                    "mrr": result["mrr"],
                })

In [98]:
import pandas as pd

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="mrr",
    ascending=False
)

print(results_df.head(10))

   code  name  symbol  countries_text  hit_rate       mrr
0   1.0   1.0     0.5             1.0      0.94  0.932222
1   1.0   1.0     0.5             2.0      0.94  0.932222
2   1.0   1.0     0.5             5.0      0.94  0.932222
3   1.0   1.0     1.0             1.0      0.94  0.932222
4   1.0   1.0     1.0             2.0      0.94  0.932222
5   1.0   1.0     1.0             5.0      0.94  0.932222
6   1.0   1.0     2.0             1.0      0.94  0.932222
7   1.0   1.0     2.0             2.0      0.94  0.932222
8   1.0   1.0     2.0             5.0      0.94  0.932222
9   1.0   2.0     0.5             1.0      0.94  0.932222


In [100]:
import pandas as pd

df_ground_truth = pd.read_csv("../data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [101]:
from ingest import load_currency_data, build_index

documents = load_currency_data()

documents_germany = [
    doc for doc in documents
    if "Germany" in doc["countries"]
]

index = build_index(documents_germany)

In [102]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["doc_id"]] = doc

In [103]:
q = ground_truth[10]
q

{'question': 'What is the currency code for the British Pound?',
 'document': 'GBP'}

In [104]:
doc_idx[q['document']]

{'code': 'GBP',
 'name': 'British Pound',
 'symbol': '£',
 'decimal_digits': 2,
 'countries': ['United Kingdom'],
 'doc_id': 'GBP',
 'countries_text': 'United Kingdom'}

In [107]:
from evaluation_utils import CurrencyRAG

assistant = CurrencyRAG(
    index=index,
    llm_client=openai_client,
    model="qwen-max",
    currency_file="documents/currencies.json",
)

In [108]:
q['question']

'What is the currency code for the British Pound?'

In [109]:
answer = assistant.rag(q['question'])

In [110]:
assistant.total_cost()

0

In [111]:
print(answer)

I don't know.


In [113]:
doc_id = q["document"]
original_doc = doc_idx[doc_id]

answer_orig = {
    "code": original_doc["code"],
    "name": original_doc["name"],
    "symbol": original_doc["symbol"],
    "countries": original_doc["countries"],
}

answer_orig

{'code': 'GBP',
 'name': 'British Pound',
 'symbol': '£',
 'countries': ['United Kingdom']}

In [115]:
rag_result = {
    "question": q.get("question", q.get("query")),
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'What is the currency code for the British Pound?',
 'answer_orig': {'code': 'GBP',
  'name': 'British Pound',
  'symbol': '£',
  'countries': ['United Kingdom']},
 'document': 'GBP'}

In [116]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]

    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)

    answer_orig = {
        "code": original_doc["code"],
        "name": original_doc["name"],
        "symbol": original_doc["symbol"],
        "countries": original_doc["countries"],
    }

    result = {
        "question": question,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [117]:
record = generate_rag_answer(q)
record

{'question': 'What is the currency code for the British Pound?',
 'answer_orig': {'code': 'GBP',
  'name': 'British Pound',
  'symbol': '£',
  'countries': ['United Kingdom']},
 'document': 'GBP'}

In [118]:
assistant.total_cost()

0

In [119]:
assistant.reset_usage()

In [120]:
assistant.total_cost()

0

In [121]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [ ]:
results[:10]

In [124]:
from evaluation_utils import CurrencyRAG

assistant = CurrencyRAG(
    index=index,
    llm_client=openai_client,
    model="openai/gpt-oss-120b",
    currency_file="documents/currencies.json",
)

In [125]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

100%|██████████| 150/150 [00:01<00:00, 144.44it/s]


In [ ]:
results[:10]

In [127]:
df_results = pd.DataFrame(results)

In [129]:
df_results.head(10)

,question,answer_orig,document
0,What is the three-letter code for the US Dollar?,"{'code': 'USD', 'name': 'US Dollar', 'symbol':...",USD
1,Which symbol is used to represent the US Dollar?,"{'code': 'USD', 'name': 'US Dollar', 'symbol':...",USD
2,How many decimal digits are typically shown fo...,"{'code': 'USD', 'name': 'US Dollar', 'symbol':...",USD
3,Name three countries where the US Dollar is us...,"{'code': 'USD', 'name': 'US Dollar', 'symbol':...",USD
4,What is the doc_id associated with the US Doll...,"{'code': 'USD', 'name': 'US Dollar', 'symbol':...",USD
5,What is the three-letter code for the Euro?,"{'code': 'EUR', 'name': 'Euro', 'symbol': '€',...",EUR
6,Which symbol is used to represent the Euro cur...,"{'code': 'EUR', 'name': 'Euro', 'symbol': '€',...",EUR
7,How many decimal digits does the Euro use?,"{'code': 'EUR', 'name': 'Euro', 'symbol': '€',...",EUR
8,List three countries that use the Euro as thei...,"{'code': 'EUR', 'name': 'Euro', 'symbol': '€',...",EUR
9,What is the name of the currency with the code...,"{'code': 'EUR', 'name': 'Euro', 'symbol': '€',...",EUR


In [130]:
assistant.total_cost()

0.0001983

In [133]:
from pathlib import Path

output_dir = Path("data")
output_dir.mkdir(parents=True, exist_ok=True)

df_results.to_csv(output_dir / "rag-answers-new.csv", index=False)

In [135]:
df_results.to_csv("data/rag-answers-new.csv", index=False)

In [136]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [137]:
aqa_judge_instructions = """
You are an expert evaluator of currency-related answers.

Compare the AI answer with the ground-truth currency information.

Rules:
- Accept answers with equivalent meaning, even if worded differently.
- Check currency names, codes, symbols, and country associations.
- Ignore style differences and irrelevant details.
- Mark 'bad' only if the answer is factually incorrect or misses
  the key information needed to answer the question.
- Focus on correctness, not wording.
""".strip()

In [139]:
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

In [140]:
df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [141]:
rec = answers[0]
rec

{'question': 'What is the three-letter code for the US Dollar?',
 'answer_orig': "{'code': 'USD', 'name': 'US Dollar', 'symbol': '$', 'countries': ['United States', 'Ecuador', 'El Salvador']}",
 'document': 'USD'}

In [187]:
aqa_judge_prompt = """
Question:
{question}

Original Currency Information (ground truth):
{answer_orig}
""".strip()

In [188]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
)

print(prompt)

Question:
What is the three-letter code for the US Dollar?

Original Currency Information (ground truth):
{'code': 'USD', 'name': 'US Dollar', 'symbol': '$', 'countries': ['United States', 'Ecuador', 'El Salvador']}


In [189]:
import time

In [158]:
def llm_structured_retry(
    client,
    instructions,
    user_prompt,
    output_type,
    model="openai/gpt-oss-120b",
    max_retries=3,
):
    for attempt in range(max_retries):
        try:
            return llm_structured(
                client=client,
                instructions=instructions,
                user_prompt=user_prompt,
                output_type=output_type,
                model=model,
            )
        except Exception:
            if attempt == max_retries - 1:
                raise

            time.sleep(2 ** attempt)

In [159]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning="No answer was provided to evaluate against the ground truth. Since the required three-letter code 'USD' was not given, the response is incomplete and factually missing the key information.", score='bad')

In [160]:
calc_price(usage)

{'input_cost': 3.66e-05,
 'output_cost': 9.520000000000001e-05,
 'total_cost': 0.0001318}

In [190]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
)

eval_result

AnswerEvaluation(reasoning='The AI did not provide any answer to the question asking for the three-letter code for the US Dollar, so it fails to supply the required information (USD). Therefore it is incorrect.', score='bad')

In [191]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [192]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=1) as pool:
    results = map_progress(pool, answers, judge_record)

100%|██████████| 150/150 [08:14<00:00,  3.30s/it]


In [183]:
def evaluate_aqa(question, answer_orig, model="openai/gpt-oss-120b"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [193]:
rec

{'question': 'What is the three-letter code for the US Dollar?',
 'answer_orig': "{'code': 'USD', 'name': 'US Dollar', 'symbol': '$', 'countries': ['United States', 'Ecuador', 'El Salvador']}",
 'document': 'USD'}

In [194]:
results[10]

({'question': 'What is the currency code for the British Pound?',
  'document': 'GBP',
  'score': 'bad',
  'reasoning': 'No answer was provided to evaluate against the ground truth. Since the required information (the currency code "GBP") is missing, the response is incomplete and therefore incorrect.'},
 ResponseUsage(input_tokens=358, input_tokens_details=InputTokensDetails(cache_write_tokens=None, cached_tokens=0), output_tokens=318, output_tokens_details=OutputTokensDetails(reasoning_tokens=263), total_tokens=676))

In [195]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [196]:
calc_total_price(usages)

0.025333900000000003

In [197]:
df_eval = pd.DataFrame(evaluations)

In [198]:
df_eval

,question,document,score,reasoning
0,What is the three-letter code for the US Dollar?,USD,bad,No answer was provided to the question. The re...
1,Which symbol is used to represent the US Dollar?,USD,bad,No answer was provided to evaluate against the...
2,How many decimal digits are typically shown fo...,USD,bad,No answer was provided to evaluate against the...
3,Name three countries where the US Dollar is us...,USD,bad,The AI did not provide any response naming thr...
4,What is the doc_id associated with the US Doll...,USD,bad,The AI did not provide any answer to the quest...
...,...,...,...,...
145,What is the three-letter code for the Saudi Ri...,SAR,bad,No answer was provided to the question asking ...
146,Which country uses the currency with the symbo...,SAR,bad,No answer was provided to evaluate against the...
147,What is the name of the currency that has the ...,SAR,bad,"No answer was provided to the question, so the..."
148,How many decimal digits does the Saudi Riyal h...,SAR,bad,"No answer was provided to the question, so it ..."


In [199]:
df_eval.score.value_counts()

score
bad     146
good      4
Name: count, dtype: int64

In [200]:
df_eval.score.value_counts(normalize=True)

score
bad     0.973333
good    0.026667
Name: proportion, dtype: float64